# Counterfactuals for Binary Classification

In [1]:
import seaborn as sns
import warnings
# import time
import joblib
from tqdm import tqdm

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import random

from sklearn.model_selection import train_test_split

import dice_ml
from dice_ml import Dice

# #. from module.dataload import DPN_data
# #. from module.eda import EDA
# #. from module.eda import EDAHelper

In [2]:
import sys 
sys.path.append('..')  

# from module.backends.backend_adapter import get_dice_components
from module.dataload import DPN_data
# from module.eda import EDA


In [3]:
warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

## Preparations

### Data Loading

In [5]:
D = DPN_data("../dataset/Sudoscan Working File with Stats.xlsx")
D.load(classification="binary")
D.load().tail(2)

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
188,0,60.0,1,5.0,1.0,12.20,1,0,1,0,0,1,1,1,1,8.0,5.03,37.9,0.0,0.0,36.3,4.85,5.05,3.06,53.1,5.58,37.2,0.0,0.0,32.2,4.2,5.09,3.28,53.5,16.0,11.0,21.0,9.0,46.0,32.0,1
189,0,65.0,1,15.0,1.0,7.59,1,1,0,1,0,1,1,1,1,8.0,0.00,0.0,0.0,0.0,43.2,5.80,0.56,0.20,0.0,0.00,0.0,0.0,0.0,48.1,5.7,0.27,0.11,0.0,39.0,16.0,41.0,23.0,43.0,44.0,1


Binary Classification Classes:  ['Negative', 'Possible', 'Probable'] vs 'Confirmed'


In [6]:
df = D.df
data_cols = df.drop(D.non_data_cols, axis=1, errors="ignore").columns
len(data_cols), data_cols

(40,
 Index(['SEX', 'AGE', 'SUBJ', 'DM_DUR', 'INSULIN', 'HBA1C', 'HPN', 'PAOD',
        'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR',
        'MNSI', 'SSA_L', 'SSC_L', 'SPSA_L', 'SPSC_L', 'MCV_L', 'DL_L',
        'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L', 'SSA_R', 'SSC_R', 'SPSA_R',
        'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R', 'FWAVE_R',
        'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM',
        'NS', 'CAS'],
       dtype='object'))

### Data Inspection

In [7]:
X = df[data_cols]
y = df['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

### Define Metrics

In [8]:
import numpy as np
import pandas as pd
# from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, make_scorer
)

# Custom Youden Index scorer
def youden_index_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return sensitivity + specificity - 1

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

youden_scorer = make_scorer(youden_index_score, greater_is_better=True)
specificity_scorer = make_scorer(specificity_score)

### Load optimized model from pickle file

In [9]:
rf_optimized_results = joblib.load(rf"outputs\all_features\random_forest.pkl")

model_name = rf_optimized_results["name"]
best_params = rf_optimized_results["best_params"]
best_score = rf_optimized_results["best_score"]  
optimized_model = rf_optimized_results["optimized_model"] 
optimized_model_metrics = rf_optimized_results["optimized_model_metrics"]

optimized_model_metrics

Accuracy                0.938
Precision               0.969
Recall (Sensitivity)    0.939
Specificity             0.933
F1                      0.954
ROC-AUC                 0.992
Youden Index            0.873
dtype: float64

In [10]:
rf_optimized_results

{'name': 'random_forest',
 'best_params': OrderedDict([('bootstrap', False),
              ('class_weight', 'balanced_subsample'),
              ('criterion', 'gini'),
              ('max_depth', 25),
              ('max_features', 'sqrt'),
              ('min_samples_leaf', 1),
              ('min_samples_split', 7),
              ('n_estimators', 62)]),
 'best_score': 0.9839743589743591,
 'optimized_model': RandomForestClassifier(bootstrap=False, class_weight='balanced_subsample',
                        max_depth=25, min_samples_split=7, n_estimators=62,
                        n_jobs=-1, random_state=42),
 'optimized_model_metrics': Accuracy                0.938
 Precision               0.969
 Recall (Sensitivity)    0.939
 Specificity             0.933
 F1                      0.954
 ROC-AUC                 0.992
 Youden Index            0.873
 dtype: float64}

### Model Evaluation

#### Set Global Variables

In [11]:
test_size = 0.25
verbosity = 1

#### Train Test Split

In [12]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=0, stratify=y)

In [13]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall (Sensitivity)": recall_score(y_test, y_pred),
        "Specificity": specificity_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
        "Youden Index": youden_index_score(y_test, y_pred)
    }
    return pd.Series(metrics)

In [14]:
# optimized_model.fit(X_train, y_train) # no longer needed
# y_pred = optimized_model.predict(X_val)
# y_proba = optimized_model.predict_proba(X_val)[:, 1]
results = evaluate_model(optimized_model, X_val, y_val)
print(results)

Accuracy                0.938
Precision               0.969
Recall (Sensitivity)    0.939
Specificity             0.933
F1                      0.954
ROC-AUC                 0.992
Youden Index            0.873
dtype: float64


### Define Features Group Color Map

In [15]:
import matplotlib.patches as mpatches

palette = {
    'Teal': '#8DD3C7', 
    'Yellow': '#FFFFB3', 
    'Lavender': '#BEBADA', 
    'Coral': '#FB8072', 
    'Blue': '#80B1D3', 
    'Orange': '#FDB462', 
    'Green': '#B3DE69', 
    'Pink': '#FCCDE5', 
    'Purple': '#BC80BD', 
    'Gray': '#D9D9D9', 
    'Red': '#E41A1C' 
}

COLOR_GROUP_MAP = {
    'Nerve Conduction Studies': palette['Blue'],
    'Sudoscan': palette['Orange'],
    'Profile': palette['Teal'],
    'Comorbidities': palette['Red'],
    'Neurology Examination': palette['Pink'],
    'MNSI': palette['Green'],
    # 'Others': palette['Gray']
}

def get_colors(labels):
    # D is assumed to be available in the scope (e.g., imported module or class instance)
    return [
        COLOR_GROUP_MAP['Nerve Conduction Studies']  if label in D.ncs_cols else 
        COLOR_GROUP_MAP['Sudoscan'] if label in D.sudo_cols else 
        COLOR_GROUP_MAP['Profile'] if label in D.profile_cols else 
        COLOR_GROUP_MAP['Comorbidities'] if label in D.comorbidity_cols else # Assuming 'Red' is the intended color
        COLOR_GROUP_MAP['Neurology Examination'] if label in D.neuro_cols else 
        COLOR_GROUP_MAP['MNSI'] if label in D.mnsi_col else 
        palette['Gray']
        for label in labels
    ]

## Local Counterfactual Analysis

### Preparation

#### Setup

In [16]:
dfXy = pd.concat([X, y], axis=1)
X.shape, y.shape, dfXy.shape

((190, 40), (190,), (190, 41))

####  Define Actionable, Progressive, Immutable Features

In [17]:
allfeature_cols = dfXy.columns.drop('Confirmed_Binary_DPN').to_list()
continuous_cols = dfXy.columns.difference(D.categorical_cols+['Confirmed_Binary_DPN']).to_list()

actionable_cols = ['HBA1C', 'DSLPDMIA', 'INSULIN']
progressive_cols = ['AGE', 'DM_DUR', 'HPN', 'PAOD', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI']
immutable_cols = ["SEX"]

# Feature list from data loader
# profile_cols = ['SEX', 'AGE', 'SUBJ', 'DM_DUR', 'INSULIN', 'HBA1C']
#     comorbidity_cols = ['HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS']
#     neuro_cols = ['DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR']
#     mnsi_col = ['MNSI']
#     ncs_cols = ['SSA_L', 'SSC_L', 'SPSA_L', 'SPSC_L', 'MCV_L', 'DL_L', 'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L',
#                 'SSA_R', 'SSC_R', 'SPSA_R', 'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R', 'FWAVE_R']
#     sudo_cols = ['FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'NS', 'CAS']

In [18]:
print('all feature columns:\n', len(allfeature_cols), allfeature_cols)
print('categorical columns:\n', len(D.categorical_cols), D.categorical_cols)
print('continuous_columns:\n', len(continuous_cols), continuous_cols)

all feature columns:
 40 ['SEX', 'AGE', 'SUBJ', 'DM_DUR', 'INSULIN', 'HBA1C', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI', 'SSA_L', 'SSC_L', 'SPSA_L', 'SPSC_L', 'MCV_L', 'DL_L', 'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L', 'SSA_R', 'SSC_R', 'SPSA_R', 'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R', 'FWAVE_R', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'NS', 'CAS']
categorical columns:
 12 ['SEX', 'SUBJ', 'INSULIN', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR']
continuous_columns:
 28 ['AGE', 'CAS', 'CMAPANK_L', 'CMAPANK_R', 'CMAPKNE_L', 'CMAPKNE_R', 'DL_L', 'DL_R', 'DM_DUR', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'FWAVE_L', 'FWAVE_R', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'HBA1C', 'MCV_L', 'MCV_R', 'MNSI', 'NS', 'SPSA_L', 'SPSA_R', 'SPSC_L', 'SPSC_R', 'SSA_L', 'SSA_R', 'SSC_L', 'SSC_R']


In [19]:
set(D.categorical_cols) & set(progressive_cols)

{'CKD', 'DEC_AR', 'DEC_LTS', 'DEC_PPS', 'DEC_VS', 'GBS', 'HPN', 'PAOD'}

#### Global Permitted Range

In [20]:
def get_global_permitted_range(continuous_cols):
    global_permitted_range = {}
    for col in continuous_cols: # no need to set range for categorical columns
        stdev = dfXy[col].std()
        minval = dfXy[col].min()
        maxval = dfXy[col].max()
        minval = 0 if minval==0 else max(0, minval-stdev)
        maxval = maxval + stdev
        global_permitted_range[col] = [minval, maxval]
    return global_permitted_range

In [21]:
global_permitted_range = get_global_permitted_range(continuous_cols)
# visualize global_permitted_range
global_permitted_range_df = pd.DataFrame(global_permitted_range).transpose()
global_permitted_range_df.columns = ['min', 'max']
global_permitted_range_df

,min,max
AGE,8.111,100.889
CAS,0.000,58.027
CMAPANK_L,0.000,29.589
CMAPANK_R,0.000,31.434
CMAPKNE_L,0.000,22.551
CMAPKNE_R,0.000,23.008
DL_L,0.000,48.650
DL_R,0.000,16.399
DM_DUR,0.000,37.797
FEET_MEAN_ESC,0.000,106.560


#### Setup Explainer Object

In [22]:
d = dice_ml.Data(dataframe=dfXy, 
                 categorical_features = D.categorical_cols,
                 continuous_features=continuous_cols,                  
                 permitted_range = global_permitted_range,                 
                 outcome_name='Confirmed_Binary_DPN')
m = dice_ml.Model(model=optimized_model, backend="sklearn", model_type="classifier")
exp = dice_ml.Dice(d, m, method="genetic")

#### Local Permitted Range

In [23]:
# def get_local_permitted_range_old(instance, global_permitted_range, monotonic_cols):
#     local_permitted_range = global_permitted_range.copy()
#     for col in monotonic_cols:
#         if col in global_permitted_range.keys():
#             local_permitted_range[col][0] = instance.iloc[0][col] # adjust the minimum
#             print(col, local_permitted_range[col])
#     return local_permitted_range

In [24]:
def get_local_permitted_range(instance, allfeature_cols, continuous_cols, monotonic_cols):
    local_permitted_range = {}
    for col in allfeature_cols:

        if col in D.categorical_cols:
            # it does not make sense to set a range for categoricals
            continue
        
        instance_val = instance.iloc[0][col]
        col_stdev = dfXy[col].std()
        col_min = dfXy[col].min()
        col_max = dfXy[col].max()

        if col in monotonic_cols: # true for categoricals and continuous
            minval = instance_val 
        elif col in continuous_cols:
            minval = 0 if instance_val==0 else max(0, instance_val-col_stdev)    
        else: # fall back default 
            minval = col_min

        if col in continuous_cols:
            maxval = instance_val + col_stdev
        # elif col in D.categorical_cols:
        #     maxval = max(1, col_max)
        else: # fall back default 
            maxval = col_max

        local_permitted_range[col] = [minval, maxval]
    return local_permitted_range

##### Local Permitted Range for Patient Index 40

In [25]:
pidx = 40
query_instance = X[pidx:pidx+1]
instance_permitted_range = get_local_permitted_range(
    query_instance, allfeature_cols, continuous_cols, progressive_cols)

In [26]:
# visualize instance_permitted_range
instance_permitted_range_df = pd.DataFrame(instance_permitted_range).transpose()
instance_permitted_range_df.columns = ['min', 'max']
instance_permitted_range_df

,min,max
AGE,47.000,58.889
DM_DUR,14.000,21.797
HBA1C,7.615,13.045
MNSI,7.000,9.689
SSA_L,3.101,18.999
SSC_L,29.553,68.647
SPSA_L,0.000,11.474
SPSC_L,21.699,65.301
MCV_L,33.677,47.123
DL_L,0.300,6.600


In [27]:
# visualize min max with patient data
pd.concat([query_instance, instance_permitted_range_df.transpose()]).transpose()

,40,min,max
SEX,0.00,NaN,NaN
AGE,47.00,47.000,58.889
SUBJ,1.00,NaN,NaN
DM_DUR,14.00,14.000,21.797
INSULIN,1.00,NaN,NaN
HBA1C,10.33,7.615,13.045
HPN,0.00,NaN,NaN
PAOD,0.00,NaN,NaN
DSLPDMIA,1.00,NaN,NaN
CKD,0.00,NaN,NaN


##### Generate 5 Sample Local Counterfactuals for Patient 40

In [28]:
print(f"generating counterfactuals for the {model_name} model")    
e1 = exp.generate_counterfactuals(
    query_instance, total_CFs=5, 
    desired_class="opposite", 
    permitted_range=instance_permitted_range,
    features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list()
    )    
e1.visualize_as_dataframe(show_only_changes=True) 

generating counterfactuals for the random_forest model


100%|██████████| 1/1 [00:01<00:00,  1.12s/it]

Query instance (original outcome : 0)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,0,47.0,1,14.0,1.0,10.33,0,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0,0



Diverse Counterfactual set (new outcome: 1)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,-,-,-,-,-,10.5,1.0,-,-,-,-,-,-,-,-,-,11.8,45.5,7.1,42.2,-,5.2,2.31,1.3,-,13.2,45.2,7.6,42.9,-,3.5,4.88,-,-,75.0,2.0,78.0,-,-,-,1.0
0,-,-,-,-,0.0,10.3,-,-,0.0,-,-,-,-,-,-,-,12.7,54.4,0.0,51.8,-,2.9,-,-,43.9,12.2,40.4,11.4,22.4,-,3.2,9.27,8.3,45.2,63.4,-,-,0.0,-,-,1.0
0,-,-,-,-,-,8.4,-,-,-,-,-,-,-,-,-,-,16.6,-,3.6,51.8,45.2,2.9,7.52,5.31,45.5,16.1,40.4,5.0,51.3,43.9,3.2,9.27,7.34,45.2,-,-,78.0,10.0,-,-,1.0
0,-,-,-,-,0.0,10.3,1.0,-,-,-,-,1.0,1.0,-,1.0,-,7.6,45.2,4.7,21.7,33.7,5.05,5.51,-,-,8.5,40.3,0.0,41.0,34.5,3.5,-,-,-,64.0,-,-,3.0,-,-,1.0
0,-,-,-,-,-,10.3,-,-,-,-,-,-,-,-,-,-,8.9,53.4,6.5,52.4,40.7,3.8,5.45,4.12,52.2,8.1,52.4,5.6,52.2,41.1,3.5,8.93,6.09,47.4,81.0,4.0,85.0,-,-,-,1.0


#### Inspect logits

In [29]:
# optimized_model.fit(X_train, y_train) # no longer needed
y_pred = optimized_model.predict(X_val)
y_proba = optimized_model.predict_proba(X_val)[:, 1]

In [30]:
y_pred

array([1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       1, 1, 0, 0])

In [31]:
y_proba

array([1.   , 0.965, 0.341, 0.037, 0.897, 0.069, 0.795, 1.   , 0.074,
       0.691, 0.952, 0.969, 0.971, 0.883, 0.609, 0.985, 0.984, 0.976,
       0.037, 0.472, 0.757, 0.959, 0.76 , 0.99 , 0.984, 0.117, 0.984,
       0.775, 0.977, 0.033, 0.069, 0.99 , 0.952, 1.   , 0.017, 0.175,
       0.492, 0.008, 1.   , 0.483, 0.729, 0.901, 0.837, 0.951, 0.962,
       0.932, 0.163, 0.299])

In [32]:
y_val

165    1
2      1
182    0
21     0
10     1
70     0
18     1
83     1
81     0
178    0
145    1
16     1
108    1
148    1
127    1
173    1
150    1
48     1
72     0
40     1
151    1
60     1
8      1
76     1
58     1
129    0
103    1
171    1
189    1
147    0
7      0
91     1
0      1
64     1
1      0
153    0
96     1
159    0
24     1
42     0
54     1
118    1
82     1
140    1
15     1
114    1
135    0
80     0
Name: Confirmed_Binary_DPN, dtype: int32

### Borderline Local Counterfactuals

#### Generate Borderline Cases

In [33]:
def get_borderline_cases(model, X, y, threshold=0.5, delta=0.1):
    """Return borderline cases around the decision threshold."""
    y_prob = model.predict_proba(X)[:, 1]
    margin = np.abs(y_prob - threshold)
    
    borderline_idx = np.where(margin <= delta)[0]
    borderline_df = X.iloc[borderline_idx].copy()
    borderline_df["pred_prob"] = y_prob[borderline_idx]
    borderline_df["margin"] = margin[borderline_idx]
    borderline_df["pred_label"] = (y_prob[borderline_idx] >= threshold).astype(int)
    borderline_df["true_label"] = y.iloc[borderline_idx].values
    borderline_df["misclassified"] = borderline_df["pred_label"] != borderline_df["true_label"]
    
    print(f"Found {len(borderline_df)} borderline cases (|p - {threshold}| ≤ {delta})")
    return borderline_df


In [36]:
borderline_df = get_borderline_cases(optimized_model, X, y, delta=0.2)
borderline_df = borderline_df[D.profile_cols+['pred_prob','margin','pred_label','true_label','misclassified']].sort_values(by='margin')
borderline_df

Found 7 borderline cases (|p - 0.5| ≤ 0.2)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,pred_prob,margin,pred_label,true_label,misclassified
96,0,51.0,1,3.0,1.0,12.40,0.492,0.008,0,1,True
42,1,65.0,1,0.0,0.0,6.90,0.483,0.017,0,0,False
40,0,47.0,1,14.0,1.0,10.33,0.472,0.028,0,1,True
127,0,35.0,1,0.0,0.0,6.30,0.609,0.109,1,1,False
182,0,35.0,0,1.0,1.0,5.00,0.341,0.159,0,0,False
68,0,64.0,1,1.0,0.0,5.50,0.687,0.187,1,1,False
178,0,57.0,1,10.0,0.0,6.10,0.691,0.191,1,0,True


In [41]:
borderline_df.to_csv(r'outputs\counterfactuals\local\binary\borderline_df_delta0.2.csv')

In [42]:
# inspect Patient 40
pidx = 40
dfXy.iloc[pidx:pidx+1]

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
40,0,47.0,1,14.0,1.0,10.33,0,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0,1


In [43]:
pidx = 40
query_instance = X[pidx:pidx+1]
instance_permitted_range = get_local_permitted_range(query_instance, allfeature_cols, continuous_cols, progressive_cols)

In [44]:
print(f"generating counterfactuals for the {model_name} model")    
e1 = exp.generate_counterfactuals(
    query_instance, total_CFs=5, 
    desired_class="opposite", 
    permitted_range=instance_permitted_range,
    features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list()
    )    
e1.visualize_as_dataframe(show_only_changes=True) 

generating counterfactuals for the random_forest model


100%|██████████| 1/1 [00:01<00:00,  1.15s/it]

Query instance (original outcome : 0)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,0,47.0,1,14.0,1.0,10.33,0,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0,0



Diverse Counterfactual set (new outcome: 1)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,-,-,-,-,-,10.5,1.0,-,-,-,-,-,-,-,-,-,11.8,45.5,7.1,42.2,-,5.2,2.31,1.3,-,13.2,45.2,7.6,42.9,-,3.5,4.88,-,-,75.0,2.0,78.0,-,-,-,1.0
0,-,-,-,-,-,8.4,-,-,-,-,-,-,-,-,-,-,16.6,-,3.6,51.8,45.2,2.9,7.52,5.31,45.5,16.1,40.4,5.0,51.3,43.9,3.2,9.27,7.34,45.2,-,-,78.0,10.0,-,-,1.0
0,-,-,-,-,-,10.3,-,-,-,-,-,-,-,-,-,-,8.9,53.4,6.5,52.4,40.7,3.8,5.45,4.12,52.2,8.1,52.4,5.6,52.2,41.1,3.5,8.93,6.09,47.4,81.0,4.0,85.0,-,-,-,1.0
0,-,-,-,-,-,9.0,-,-,0.0,-,-,-,1.0,-,1.0,-,4.9,-,0.0,-,43.2,3.5,-,-,43.3,12.1,45.3,0.0,-,44.0,3.5,9.98,7.18,42.4,76.0,0.0,76.0,-,90.0,-,1.0
0,-,-,-,-,0.0,10.3,1.0,-,0.0,-,-,1.0,1.0,-,1.0,-,6.4,59.4,4.7,52.0,34.9,5.05,0.68,-,-,6.4,55.9,6.0,50.4,35.6,3.5,-,-,-,64.0,-,75.0,-,-,-,1.0


In [45]:
e1_cfdf = e1.cf_examples_list[0].final_cfs_df
e1_cfdf

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,0,47.0,1,14.0,1.0,10.5,1,0,1,0,0,0,0,0,0,7.0,11.8,45.5,7.1,42.2,40.4,5.20,2.31,1.30,48.3,13.2,45.2,7.6,42.9,41.7,3.5,4.88,7.31,47.9,75.0,2.0,78.0,2.0,84.0,17.0,1
0,0,47.0,1,14.0,1.0,8.4,0,0,1,0,0,0,0,0,0,7.0,16.6,49.1,3.6,51.8,45.2,2.90,7.52,5.31,45.5,16.1,40.4,5.0,51.3,43.9,3.2,9.27,7.34,45.2,83.0,3.0,78.0,10.0,84.0,17.0,1
0,0,47.0,1,14.0,1.0,10.3,0,0,1,0,0,0,0,0,0,7.0,8.9,53.4,6.5,52.4,40.7,3.80,5.45,4.12,52.2,8.1,52.4,5.6,52.2,41.1,3.5,8.93,6.09,47.4,81.0,4.0,85.0,2.0,84.0,17.0,1
0,0,47.0,1,14.0,1.0,9.0,0,0,0,0,0,0,1,0,1,7.0,4.9,49.2,0.0,43.5,43.2,3.50,4.85,3.99,43.3,12.1,45.3,0.0,44.0,44.0,3.5,9.98,7.18,42.4,76.0,0.0,76.0,2.0,90.0,17.0,1
0,0,47.0,1,14.0,0.0,10.3,1,0,0,0,0,1,1,0,1,7.0,6.4,59.4,4.7,52.0,34.9,5.05,0.68,3.99,48.3,6.4,55.9,6.0,50.4,35.6,3.5,8.07,7.31,47.9,64.0,3.0,75.0,2.0,84.0,17.0,1


### Prototypical and Atypical

##### Prototypical (most representative)

##### Atypical (deviating from standard/common)

### Generate Local Counterfactuals

#### generate_diverse_cfs Function

In [46]:
def generate_diverse_cfs(dice_exp, instance, total_CFs=30, features_to_vary='all', permitted_range={}, seeds=[0,1,2,3,4], diversity_weight=1.5):
    """Generate diverse counterfactuals across multiple seeds."""
    all_cfs = []
    for s in seeds:
        # manually set random seed
        np.random.seed(s)
        random.seed(s) 

        cf = dice_exp.generate_counterfactuals(
            instance,
            total_CFs=total_CFs,
            desired_class="opposite",
            features_to_vary=features_to_vary,
            permitted_range=permitted_range,
            #random_seed=s,
            diversity_weight=diversity_weight
        )        
        df_cf = cf.cf_examples_list[0].final_cfs_df
        if not df_cf.empty:
            all_cfs.append(df_cf)
    if all_cfs:
        combined = pd.concat(all_cfs).drop_duplicates().reset_index(drop=True)
        return combined
    else:
        return pd.DataFrame()

#### Patient Index 40 (Patient 41 in Spreadsheet)

##### Generate Counterfactuals

In [47]:
pidx = 40 # dataframe is zero-indexed
query_instance = X[pidx:pidx+1]
df_dcf = generate_diverse_cfs(
    exp,
    query_instance, total_CFs=30,
    permitted_range=instance_permitted_range,
    features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list()
    )

100%|██████████| 1/1 [00:07<00:00,  7.76s/it]


In [51]:
df_dcf.tail(5)

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
51,0,47.0,1,14.0,0.0,10.3,1,0,1,1,0,1,0,1,1,7.0,11.1,29.6,0.0,43.5,33.7,5.30,3.18,3.99,48.3,12.1,43.7,0.0,44.0,41.7,3.2,8.07,7.31,47.9,63.4,0.0,87.0,0.0,84.0,17.0,1
52,0,47.0,1,14.0,0.0,10.3,0,0,0,0,0,1,1,0,1,7.0,3.1,49.1,0.0,21.7,33.7,3.45,4.85,3.99,54.4,3.3,43.7,0.0,44.0,36.2,3.5,8.07,7.31,47.9,83.0,3.0,68.0,2.0,68.1,8.0,1
53,0,47.0,1,14.0,0.0,10.3,0,1,1,0,0,0,1,1,1,9.0,11.8,49.1,0.0,43.5,33.7,3.45,4.85,3.99,48.3,12.1,45.2,7.6,22.4,41.7,3.5,4.88,4.00,47.9,83.0,2.0,68.0,2.0,84.0,17.0,1
54,0,47.0,1,14.0,0.0,10.3,0,0,0,0,0,1,1,1,1,7.0,11.1,49.1,4.7,43.5,34.9,3.45,0.06,0.61,48.3,6.4,24.7,0.0,44.0,35.6,3.5,8.07,7.31,47.9,83.0,0.0,75.0,2.0,84.0,17.0,1
55,0,48.0,1,14.0,0.0,10.3,0,0,0,1,0,0,1,0,0,7.0,14.4,49.1,0.0,43.5,40.4,4.30,7.06,0.61,52.8,12.1,43.7,0.0,44.0,42.1,3.5,3.38,4.04,54.2,83.0,3.0,87.0,2.0,84.0,17.0,1


In [52]:
dfXy.columns.__len__()

41

In [53]:
df_dcf.columns.__len__()

41

In [104]:
n = 55
save_every = 15
[(b, b*15 + save_every) for b in range(int(np.ceil(n/save_every))) ]

[(0, 15), (1, 30), (2, 45), (3, 60)]

In [ ]:
np.ceil(3.4).to

(4, 1)

In [101]:
np.ceil?

Signature:       np.ceil(*args, **kwargs)
Type:            ufunc
String form:     <ufunc 'ceil'>
File:            c:\users\acbriza\anaconda3\envs\dpncf\lib\site-packages\numpy\__init__.py
Docstring:      
ceil(x, /, out=None, *, where=True, casting='same_kind', order='K', dtype=None, subok=True[, signature, extobj])

Return the ceiling of the input, element-wise.

The ceil of the scalar `x` is the smallest integer `i`, such that
``i >= x``.  It is often denoted as :math:`\lceil x \rceil`.

Parameters
----------
x : array_like
    Input data.
out : ndarray, None, or tuple of ndarray and None, optional
    A location into which the result is stored. If provided, it must have
    a shape that the inputs broadcast to. If not provided or None,
    a freshly-allocated array is returned. A tuple (possible only as a
    keyword argument) must have length equal to the number of outputs.
where : array_like, optional
    This condition is broadcast over the input. At locations where the
    condi

In [ ]:
def plot_local_cf_heatmap(df_dcf, query_instance, 
                          pstr, pred, actual, 
                          savedir,
                          save_every=15,  
                          figsize=(15, 6.5)
                          ):   
    # Compute differences (each row in df_large vs. the single row)
    diffs = df_dcf - query_instance.iloc[0]
    print("diffs.shape: ", diffs.shape)
    batch_ranges = [(b*save_every, b*save_every+save_every) for b in range(int(np.ceil(df_dcf.shape[0]/save_every))) ]
    print("batch_ranges: ",  batch_ranges)
    for idx_start, idx_end in batch_ranges:
        print("idx_start, idx_end: ", idx_start, idx_end)
        diff = diffs.iloc[idx_start: idx_end]
        print("diff.shape: ",  diff.shape)
        diff = diff[dfXy.drop('Confirmed_Binary_DPN', axis=1).columns]
        print("diff.shape: ",  diff.shape)

        # Create mask where values == 0
        mask = diff == 0

        # Plot heatmap
        plt.figure(figsize=figsize)
        ax = sns.heatmap(
            diff,
            mask=mask,            # hide zero differences
            cmap="RdBu",        # diverging color map centered at 0
            center=0,
            annot=True,           # show annotations
            fmt=".2f",            # format annotations to 2 decimal places
            annot_kws={"size": 6},# smaller font
            cbar_kws={'label': 'Difference'}
        )

        ax.set_yticks(np.arange(len(diff)) + 0.5)
        ax.set_yticklabels(diff.index, rotation=0, fontsize=8)

        # ✅ Make x-tick labels smaller too
        ax.set_xticklabels(diff.columns, rotation=45, ha='right', fontsize=8)

        # plt.title("Differences from Instance", fontsize=12)
        plt.title(f"Counterfactuals for Patient {pstr}: predicted {pred}, actual {actual}", fontsize=12, pad=20)
        plt.xlabel("Features")
        plt.ylabel("Counterfactuals")

        # 1. Get the feature values from the query instance
        desired_order = diff.columns.tolist() 
        query_instance_reordered = query_instance[desired_order]

        query_values = query_instance_reordered.iloc[0].values
        NEW_Y_POSITION = -0.05 

        # 2. Loop through each feature to place the value text
        for i, value in enumerate(query_values):
            # i + 0.5 centers the text in the column cell.
            # y = -0.3: Increased separation from the heatmap for visual clarity/alignment.
            ax.text(
                x=i + 0.5,
                y=NEW_Y_POSITION, #-0.3, # Adjusted from -0.2 to -0.3
                s="0" if value==0 else "1" if value==1 else f"{value:.2f}",  # Display the value, formatted
                ha='center',
                va='bottom',
                fontsize=6,
                # fontweight='bold',
                color='#1a1a1a' # Slightly darker color for visibility
            )

        # 3. Add a row header label for the new values
        ax.text(
            x=-0.5, # Position to the left of the Y-axis labels
            y=NEW_Y_POSITION, #-0.3, # Adjusted from -0.2 to -0.3
            s="Instance Values:",
            ha='right',
            va='bottom',
            fontsize=6,
            # fontweight='bold',
            color='#1a1a1a' # Slightly darker color for visibility
        )

        plt.tight_layout()
        os.makedirs(os.path.join(savedir, pstr), exist_ok=True)
        idx_end = min(idx_end, idx_start+diff.shape[0])
        fullfilepath = os.path.join(savedir, pstr, f"local_counterfactual_{pstr}_idx{idx_start}-idx{idx_end-1}.png")
        plt.savefig(fullfilepath)


In [171]:
plot_local_cf_heatmap(df_dcf, query_instance, 
                      pstr="040", pred=0, actual=1, 
                      savedir=r"outputs\counterfactuals\local\binary" , save_every=15, figsize=(15, 6.5))

0 15
15 30
30 45
45 60


##### Get Most Changed Features

In [191]:
def get_most_changed_feature(df_cf, instance):
    # Boolean mask: True if feature changed compared to the original instance
    changed_mask = df_cf.ne(instance.iloc[0])

    # Count how many counterfactuals changed each feature
    change_counts = changed_mask.sum()
    change_counts = change_counts.sort_values(ascending=False)
    return change_counts

most_changed_features = get_most_changed_feature(df_dcf, query_instance)
most_changed_features

sparsity                56
L2_dist                 56
SSA_R                   56
SSA_L                   56
L1_dist                 56
SPSA_R                  56
Confirmed_Binary_DPN    56
HBA1C                   56
DL_R                    56
SPSA_L                  56
CMAPANK_L               48
DEC_AR                  41
DEC_VS                  40
DL_L                    39
DEC_LTS                 37
DEC_PPS                 35
CMAPKNE_L               33
FEET_PCT_ASYM           32
MCV_L                   28
INSULIN                 27
HPN                     27
HAND_PCT_ASYM           26
DSLPDMIA                26
MCV_R                   24
FWAVE_R                 22
FWAVE_L                 20
SSC_R                   14
CKD                     14
AGE                     13
MNSI                    12
SSC_L                   11
CMAPANK_R               11
CMAPKNE_R               10
CAS                     10
HAND_MEAN_ESC           10
NS                       7
DM_DUR                   6
S

In [56]:
most_changed_features = most_changed_features.drop('Confirmed_Binary_DPN')
most_changed_features

SSA_R            56
SPSA_L           56
DL_R             56
HBA1C            56
SPSA_R           56
SSA_L            56
CMAPANK_L        48
DEC_AR           41
DEC_VS           40
DL_L             39
DEC_LTS          37
DEC_PPS          35
CMAPKNE_L        33
FEET_PCT_ASYM    32
MCV_L            28
INSULIN          27
HPN              27
HAND_PCT_ASYM    26
DSLPDMIA         26
dtype: int64

##### Analyze Sparsity and L1, L2 Distances

In [57]:
def analyze_local_cf(instance_df, cf_df, feature_costs=None):
    """
    Compute distances, sparsity, and feasibility per counterfactual.
    feature_costs: optional dict of feature->cost weights
    """
    if cf_df.empty:
        return pd.DataFrame()

    x0 = instance_df.iloc[0]
    diffs = cf_df.sub(x0)
    sparsity = (diffs != 0).sum(axis=1)   # number of columns altered
    l1 = np.abs(diffs).sum(axis=1)        
    l2 = np.sqrt((diffs**2).sum(axis=1))

    cf_df["sparsity"] = sparsity
    cf_df["L1_dist"] = l1
    cf_df["L2_dist"] = l2

    if feature_costs:
        cf_df["cost"] = sum(np.abs(diffs[f]) * feature_costs.get(f, 1) for f in diffs.columns)

    cf_df.sort_values("L1_dist").reset_index(drop=True)
    
    # generate a dataframe with the diffs and the analysis
    diffs = cf_df.drop(columns=['sparsity', 'L1_dist', 'L2_dist']).sub(x0)     
    diffs = pd.concat([diffs, cf_df[['sparsity', 'L1_dist', 'L2_dist']]], axis=1)
    return diffs,  cf_df

In [58]:
diffs, cf_ana = analyze_local_cf(query_instance, df_dcf)

In [59]:
diffs.head()

,AGE,CAS,CKD,CMAPANK_L,CMAPANK_R,CMAPKNE_L,CMAPKNE_R,Confirmed_Binary_DPN,DEC_AR,DEC_LTS,DEC_PPS,DEC_VS,DL_L,DL_R,DM_DUR,DSLPDMIA,FEET_MEAN_ESC,FEET_PCT_ASYM,FWAVE_L,FWAVE_R,GBS,HAND_MEAN_ESC,HAND_PCT_ASYM,HBA1C,HPN,INSULIN,MCV_L,MCV_R,MNSI,NS,PAOD,SEX,SPSA_L,SPSA_R,SPSC_L,SPSC_R,SSA_L,SSA_R,SSC_L,SSC_R,SUBJ,sparsity,L1_dist,L2_dist
0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,NaN,1.0,1.0,1.0,1.0,0.00,0.05,0.0,-1.0,0.0,-3.0,0.0,0.0,0.0,0.0,0.0,-0.03,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,-3.61,-5.02,0.0,0.0,0.05,-0.04,0.0,0.0,0.0,14,18.80,7.499
1,0.0,0.0,0.0,-1.33,0.0,0.00,0.0,NaN,1.0,0.0,0.0,1.0,0.00,0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.03,0.0,0.0,-6.7,0.7,2.0,0.0,0.0,0.0,-3.61,-5.02,0.0,-21.6,0.05,-0.04,0.0,0.0,-1.0,15,44.13,23.642
2,0.0,0.0,1.0,0.00,0.0,0.00,0.0,NaN,1.0,1.0,1.0,1.0,0.00,0.05,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,-0.93,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.61,-5.02,0.0,0.0,0.05,-0.04,0.0,0.0,0.0,15,24.70,10.493
3,9.0,0.0,0.0,-1.84,0.0,-1.54,0.0,NaN,0.0,1.0,1.0,0.0,1.75,0.05,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,1.47,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.61,-5.02,0.0,0.0,0.05,-0.04,0.0,0.0,0.0,15,33.37,13.008
4,0.0,7.0,0.0,-4.47,0.0,0.00,0.0,NaN,1.0,1.0,1.0,1.0,0.00,0.05,0.0,0.0,0.0,2.0,0.0,0.0,0.0,-13.0,6.0,0.77,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.61,-5.02,0.0,0.0,0.05,-0.04,0.0,0.0,0.0,17,47.01,17.939


##### Visualize top 5 counterfactuals closest to instance

In [60]:
# check top 5 counterfactuals closest to instance
cf_ana[D.profile_cols+['sparsity','L1_dist','L2_dist']].head(5)

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,sparsity,L1_dist,L2_dist
0,0,47.0,1,14.0,1.0,10.3,14,18.80,7.499
1,0,47.0,0,14.0,1.0,10.3,15,44.13,23.642
2,0,47.0,1,14.0,1.0,9.4,15,24.70,10.493
3,0,56.0,1,14.0,0.0,11.8,15,33.37,13.008
4,0,47.0,1,14.0,1.0,11.1,17,47.01,17.939


#### Filter Valid Counterfactuals

In [61]:
query_instance

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS
40,0,47.0,1,14.0,1.0,10.33,0,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0


In [62]:
progressive_categorical_cols = list(set(progressive_cols) & set(D.categorical_cols))
progressive_categorical_cols

['DEC_LTS', 'PAOD', 'DEC_VS', 'DEC_AR', 'HPN', 'DEC_PPS', 'CKD', 'GBS']

In [63]:
# check progressive categorical values of instance; filter out invalid CF: those that go from 1 to 0
query_instance[progressive_categorical_cols]
# all are 0, so all are valid counterfactuals; nothing to filter 


,DEC_LTS,PAOD,DEC_VS,DEC_AR,HPN,DEC_PPS,CKD,GBS
40,0,0,0,0,0,0,0,0


In [64]:
cf_ana

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN,sparsity,L1_dist,L2_dist
0,0,47.0,1,14.0,1.0,10.3,0,0,0,0,0,1,1,1,1,9.0,11.1,49.1,0.0,43.5,40.4,3.45,4.85,3.99,48.3,12.1,43.7,0.0,44.0,41.7,3.5,8.07,7.31,47.9,83.0,0.0,87.0,2.0,84.0,17.0,1,14,18.80,7.499
1,0,47.0,0,14.0,1.0,10.3,0,0,1,0,0,1,0,0,1,9.0,11.1,49.1,0.0,43.5,33.7,3.45,3.52,3.99,48.3,12.1,43.7,0.0,22.4,42.4,3.5,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0,1,15,44.13,23.642
2,0,47.0,1,14.0,1.0,9.4,1,0,0,1,0,1,1,1,1,7.0,11.1,49.1,0.0,43.5,40.4,3.45,4.85,3.99,48.3,12.1,43.7,0.0,44.0,41.7,3.5,8.07,7.31,47.9,83.0,3.0,87.0,10.0,84.0,17.0,1,15,24.70,10.493
3,0,56.0,1,14.0,0.0,11.8,0,0,1,0,0,0,1,1,0,7.0,11.1,49.1,0.0,43.5,40.4,5.20,3.01,2.45,48.3,12.1,43.7,0.0,44.0,41.7,3.5,8.07,7.31,47.9,83.0,9.0,87.0,2.0,84.0,17.0,1,15,33.37,13.008
4,0,47.0,1,14.0,1.0,11.1,1,0,1,0,0,1,1,1,1,7.0,11.1,49.1,0.0,43.5,40.4,3.45,0.38,3.99,48.3,12.1,43.7,0.0,44.0,41.7,3.5,8.07,7.31,47.9,83.0,5.0,74.0,8.0,84.0,24.0,1,17,47.01,17.939
5,0,48.0,1,14.0,1.0,11.1,0,0,1,0,0,1,1,1,1,7.0,11.1,49.1,0.0,43.5,40.4,3.45,2.13,1.14,48.3,12.1,43.7,0.0,44.0,35.5,3.5,4.83,7.31,58.9,83.0,3.0,87.0,2.0,84.0,17.0,1,17,40.55,15.142
6,0,47.0,1,14.0,0.0,10.3,0,0,1,0,0,0,0,1,0,9.0,12.2,49.1,9.8,43.5,40.4,4.15,4.85,3.99,43.3,12.1,57.7,5.0,44.0,41.7,3.5,8.07,9.43,42.9,83.0,6.0,87.0,7.0,70.0,17.0,1,18,60.30,22.948
7,0,47.0,1,20.0,0.0,10.3,1,0,1,0,0,1,1,1,1,9.0,11.1,49.1,0.0,43.5,40.4,5.30,0.26,3.99,48.3,12.1,43.7,0.0,44.0,41.7,3.5,8.07,7.31,47.9,83.0,8.0,87.0,2.0,84.0,17.0,1,18,34.24,11.564
8,0,47.0,1,14.0,1.0,10.9,0,0,0,0,0,1,1,1,1,7.0,11.1,49.1,0.0,43.5,40.4,4.40,0.90,0.64,48.3,12.1,43.7,0.0,44.0,41.7,4.2,8.07,7.31,47.9,83.0,3.0,87.0,8.0,84.0,17.0,1,16,29.29,10.385
9,0,47.0,1,15.0,1.0,11.0,0,0,1,0,0,1,1,1,1,7.0,11.1,49.1,0.0,43.5,40.4,3.45,0.97,0.78,48.3,12.1,43.7,0.0,44.0,41.7,4.2,8.07,7.31,47.9,83.0,10.0,87.0,0.0,84.0,17.0,1,16,31.23,11.073


#### Sufficiency

A sufficient feature  change is one that can cause the outcome change by itself.


In [ ]:
def check_sufficiency(dice_exp, instance, all_features, permitted_range, desired_class="opposite", 
                      maxiterations=500):
    """Determine necessity and sufficiency of each feature for one instance."""
    results = {}
    for f in all_features:
        results[f] = {"sufficient": "False"}
        print(f)

        # --- Sufficiency: vary only this feature  ---
        try:
            cf_suf = dice_exp.generate_counterfactuals(
                instance, total_CFs=1, desired_class=desired_class, features_to_vary=[f], 
                permitted_range=permitted_range, maxiterations=maxiterations,
            )
            if len(cf_suf.cf_examples_list[0].final_cfs_df) > 0:
                results[f]["sufficient"] = "True"
            print(f'Successfully calculated sufficiency for {f}')
        except Exception as e:
            print(f'Error calculating sufficiency for {f}')
            print(f'{e}')
            pass
        print('sufficienct: ', results[f]["sufficient"])

    return pd.DataFrame(results).T.reset_index(names="feature")

In [ ]:
most_changed_features = most_changed_features.index.to_list()
sufficient_features = ['SSA_R', 'SSA_L', 'DL_L']
forced_timeout_features = ['HBA1C',  'DEC_AR', 'DEC_VS', 'DEC_LTS', 'DEC_PPS', 'CMAPKNE_L', 'FEET_PCT_ASYM', 'INSULIN', 'HPN', 'HAND_PCT_ASYM',  'DSLPDMIA', 'DL_R']
# Other exception: 'SPSA_R'
# No CF exception: 'SPSA_L', 'CMAPANK_L', 'MCV_L', 

#check_features = forced_timeout_features
check_features = sufficient_features
print(check_features)


In [ ]:
df_s = check_sufficiency(
    exp,
    query_instance,
    #all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list()[:3],
    all_features=check_features,
    maxiterations=2000,
    permitted_range=instance_permitted_range,
    )
df_s

In [ ]:
import time
import numpy as np
import random
import pandas as pd
from tqdm import tqdm
from multiprocessing import Process, Queue

def run_dice_in_process(queue, dice_exp, instance, total_CFs, desired_class,
                        features_to_vary, permitted_range, maxiterations):
    """Run DiCE in a separate process so it can be timed out."""
    try:
        cf_suf = dice_exp.generate_counterfactuals(
                        instance, total_CFs=total_CFs, desired_class=desired_class, features_to_vary=features_to_vary, 
                        permitted_range=permitted_range, maxiterations=maxiterations,
                    )
        queue.put(cf_suf)
    except Exception as e:
        queue.put(e)

def generate_cfs_with_timeout(timeout_sec, dice_exp, instance, total_CFs, desired_class,
                          features_to_vary, permitted_range, maxiterations):
    """Run generate_counterfactuals() with timeout control."""
    q = Queue()
    p = Process(
        target=run_dice_in_process,
        args=(q, dice_exp, instance, total_CFs, desired_class,
              features_to_vary, permitted_range, maxiterations)
    )
    p.start()
    p.join(timeout=timeout_sec)

    if p.is_alive():
        p.terminate()
        p.join()
        return None  # Timeout occurred

    result = q.get() if not q.empty() else None
    if isinstance(result, Exception):
        raise result
    return result

def check_sufficiency_with_timeout(dice_exp, instance, all_features, permitted_range, 
                      desired_class="opposite", 
                      maxiterations=500, timeout_sec=30*60):
    """Determine sufficiency of each feature for one instance."""

    results = {}
    for f in tqdm(all_features):
        results[f] = {"sufficient": "False"}
        print(f)

        try:
            cf_suf = generate_cfs_with_timeout(
                timeout_sec, dice_exp, instance, total_CFs=1, desired_class=desired_class, features_to_vary=[f], 
                permitted_range=permitted_range, maxiterations=maxiterations,
            )

            # timeout
            if cf_suf is None:
                print(f"⏰ Timeout (>{timeout_sec//60} min) for feature '{f}' — skipping to next.")
                results[f]["sufficient"] = "Timeout"
                continue

            if len(cf_suf.cf_examples_list[0].final_cfs_df) > 0:
                results[f]["sufficient"] = "True"

        except Exception as e:
            print(f'Error calculating sufficiency for {f}')
            print(f'{e}')
            results[f]["sufficient"] = "Error"
            
        print('sufficient: ', results[f]["sufficient"])

    return pd.DataFrame(results).T.reset_index(names="feature")



In [ ]:
df_s = check_sufficiency_with_timeout(
    exp,
    query_instance,
    #all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list()[:3],
    all_features=check_features,
    maxiterations=500,
    permitted_range=instance_permitted_range,
    timeout_sec=5
    )
df_s

#### Necessity

A necessary feature change is one that must be altered; without it, no counterfactual achieves the desired outcome.

In [65]:
def check_necessity(dice_exp, instance, all_features, permitted_range, desired_class="opposite", 
                    maxiterations=500, total_CFs=3, seeds=[0,1,2,3,4]):
    """Determine necessity (vary all except this one across multiple seeds) of each feature for one instance."""
    results = {}
    for f in tqdm(all_features):
        results[f] = {"necessary": False}       
        # --- Necessity: vary all except this one across multiple seeds ---
        features_wo_f = [feat for feat in all_features if feat != f]
        found_cf = False
        for seed in seeds:
        
            # manually set random seed
            np.random.seed(seed)
            random.seed(seed) 

            cf_nec = dice_exp.generate_counterfactuals(
                instance, total_CFs=3, desired_class=desired_class,
                features_to_vary=features_wo_f, permitted_range=permitted_range, 
                maxiterations=maxiterations,
                #random_seed=seed
            )
            if len(cf_nec.cf_examples_list[0].final_cfs_df) > 0:
                found_cf = True
                break

        if not found_cf:
            results[f]["necessary"] = True
    return pd.DataFrame(results).T.reset_index(names="feature")

In [66]:
df_ns = check_necessity(
    exp,
    query_instance,
    all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
    maxiterations=500,
    total_CFs=2,
    permitted_range=instance_permitted_range,
    seeds=[0,1,2,3,4], 
    )
df_ns

100%|██████████| 39/39 [00:31<00:00,  1.24it/s]


,feature,necessary
0,AGE,False
1,SUBJ,False
2,DM_DUR,False
3,INSULIN,False
4,HBA1C,False
5,HPN,False
6,PAOD,False
7,DSLPDMIA,False
8,CKD,False
9,GBS,False


### Generate Files for Counterfactuals by batch

#### Setup

In [90]:
D = DPN_data("../dataset/Sudoscan Working File with Stats.xlsx")
D.load(classification="binary")
df = D.df
data_cols = df.drop(D.non_data_cols, axis=1, errors="ignore").columns

X = df[data_cols]
y = df['Confirmed_Binary_DPN']
dfXy = pd.concat([X, y], axis=1)

allfeature_cols = dfXy.columns.drop('Confirmed_Binary_DPN').to_list()
continuous_cols = dfXy.columns.difference(D.categorical_cols+['Confirmed_Binary_DPN']).to_list()

actionable_cols = ['HBA1C', 'DSLPDMIA', 'INSULIN']
progressive_cols = ['AGE', 'DM_DUR', 'HPN', 'PAOD', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI']
immutable_cols = ["SEX"]


global_permitted_range = get_global_permitted_range(continuous_cols)

d = dice_ml.Data(dataframe=dfXy, 
                 categorical_features = D.categorical_cols,
                 continuous_features=continuous_cols,                  
                 permitted_range = global_permitted_range,                 
                 outcome_name='Confirmed_Binary_DPN')
m = dice_ml.Model(model=optimized_model, backend="sklearn", model_type="classifier")
exp = dice_ml.Dice(d, m, method="genetic")

In [91]:
dfb = pd.read_csv(r'outputs\counterfactuals\local\binary\borderline_df_delta0.2.csv', index_col=0)
pindices = dfb.index.to_list()
dfb

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,pred_prob,margin,pred_label,true_label,misclassified
96,0,51.0,1,3.0,1.0,12.40,0.492,0.008,0,1,True
42,1,65.0,1,0.0,0.0,6.90,0.483,0.017,0,0,False
40,0,47.0,1,14.0,1.0,10.33,0.472,0.028,0,1,True
127,0,35.0,1,0.0,0.0,6.30,0.609,0.109,1,1,False
182,0,35.0,0,1.0,1.0,5.00,0.341,0.159,0,0,False
68,0,64.0,1,1.0,0.0,5.50,0.687,0.187,1,1,False
178,0,57.0,1,10.0,0.0,6.10,0.691,0.191,1,0,True


In [ ]:
def plot_local_cf_heatmap(df_dcf, query_instance, 
                          pstr, pred, actual, 
                          savedir,
                          save_every=15,  
                          figsize=(15, 6.5)
                          ):   
    # Compute differences (each row in df_large vs. the single row)
    
    diffs = df_dcf - query_instance.iloc[0]
    batch_ranges = [(b*save_every, b*save_every+save_every) for b in range(int(np.ceil(df_dcf.shape[0]/save_every))) ]
    for idx_start, idx_end in batch_ranges:
        print("idx_start, idx_end: ", idx_start, idx_end)
        diff = diffs.iloc[idx_start: idx_end]
        diff = diff[dfXy.drop('Confirmed_Binary_DPN', axis=1).columns]

        # Create mask where values == 0
        mask = diff == 0

        # Plot heatmap
        plt.figure(figsize=figsize)
        ax = sns.heatmap(
            diff,
            mask=mask,            # hide zero differences
            cmap="RdBu",        # diverging color map centered at 0
            center=0,
            annot=True,           # show annotations
            fmt=".2f",            # format annotations to 2 decimal places
            annot_kws={"size": 6},# smaller font
            cbar_kws={'label': 'Difference'}
        )

        ax.set_yticks(np.arange(len(diff)) + 0.5)
        ax.set_yticklabels(diff.index, rotation=0, fontsize=8)

        # ✅ Make x-tick labels smaller too
        ax.set_xticklabels(diff.columns, rotation=45, ha='right', fontsize=8)

        # plt.title("Differences from Instance", fontsize=12)
        plt.title(f"Counterfactuals for Patient {pstr}: predicted {pred}, actual {actual}", fontsize=12, pad=20)
        plt.xlabel("Features")
        plt.ylabel("Counterfactuals")

        # 1. Get the feature values from the query instance
        desired_order = diff.columns.tolist() 
        query_instance_reordered = query_instance[desired_order]

        query_values = query_instance_reordered.iloc[0].values
        NEW_Y_POSITION = -0.05 

        # 2. Loop through each feature to place the value text
        for i, value in enumerate(query_values):
            # i + 0.5 centers the text in the column cell.
            # y = -0.3: Increased separation from the heatmap for visual clarity/alignment.
            ax.text(
                x=i + 0.5,
                y=NEW_Y_POSITION, #-0.3, # Adjusted from -0.2 to -0.3
                s="0" if value==0 else "1" if value==1 else f"{value:.2f}",  # Display the value, formatted
                ha='center',
                va='bottom',
                fontsize=6,
                # fontweight='bold',
                color='#1a1a1a' # Slightly darker color for visibility
            )

        # 3. Add a row header label for the new values
        ax.text(
            x=-0.5, # Position to the left of the Y-axis labels
            y=NEW_Y_POSITION, #-0.3, # Adjusted from -0.2 to -0.3
            s="Instance Values:",
            ha='right',
            va='bottom',
            fontsize=6,
            # fontweight='bold',
            color='#1a1a1a' # Slightly darker color for visibility
        )

        plt.tight_layout()
        os.makedirs(os.path.join(savedir, pstr), exist_ok=True)
        idx_end = min(idx_end, idx_start+diff.shape[0])
        fullfilepath = os.path.join(savedir, pstr, f"local_counterfactual_{pstr}_idx{idx_start}-idx{idx_end-1}.png")
        plt.savefig(fullfilepath)

In [208]:
import os
def generate_local_cf_reports(X, exp, dfb, pidx, 
                              savedir, 
                              features_to_vary,
                              total_CFs=30,
                              seeds=[0,1,2,3,4]):
    query_instance = X[pidx:pidx+1]
    instance_permitted_range = get_local_permitted_range(query_instance, allfeature_cols, continuous_cols, progressive_cols)

    print('Generating Counterfactuals...')
    df_dcf = generate_diverse_cfs(
        exp,
        query_instance, total_CFs=total_CFs,
        permitted_range=instance_permitted_range,
        features_to_vary=features_to_vary,
        seeds=seeds,
        )
    pstr =str(pidx).zfill(3)
    os.makedirs(os.path.join(savedir, pstr), exist_ok=True)
    df_dcf.to_csv(os.path.join(savedir, pstr, f"local_cf_patient_{pstr}.csv"))

    plot_local_cf_heatmap(df_dcf, query_instance, 
                        pstr=pstr, 
                        pred=dfb.loc[pidx].pred_label, 
                        actual=dfb.loc[pidx].true_label, 
                        savedir=savedir, save_every=15, figsize=(15, 6.5))    
    
    most_changed_features = get_most_changed_feature(df_dcf, query_instance).reset_index()
    most_changed_features.columns = ['feature', 'change count']
    most_changed_features.to_csv(os.path.join(savedir, pstr, f"local_cf_most_changed_patient_{pstr}.csv"))


#### Batch process

In [ ]:
for pidx in pindices: 
    print(f"Generating counterfactual analysis for patient {pidx}")
    generate_local_cf_reports(X, exp, dfb, pidx, 
                              savedir=r"outputs\counterfactuals\local\binary",
                              features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(), 
                              total_CFs=30,
                              seeds=[0,1,2,3,4],
                              )
    

Generating counterfactual analysis for patient 96
Generating Counterfactuals...


100%|██████████| 1/1 [00:08<00:00,  8.27s/it]


df_dcf.shape:  (63, 41)
diffs.shape:  (63, 41)
batch_ranges:  [(0, 15), (15, 30), (30, 45), (45, 60), (60, 75)]
idx_start, idx_end:  0 15
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  15 30
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  30 45
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  45 60
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  60 75
diff.shape:  (3, 41)
diff.shape:  (3, 40)
Generating counterfactual analysis for patient 42
Generating Counterfactuals...


100%|██████████| 1/1 [07:05<00:00, 425.84s/it]


df_dcf.shape:  (89, 41)
diffs.shape:  (89, 41)
batch_ranges:  [(0, 15), (15, 30), (30, 45), (45, 60), (60, 75), (75, 90)]
idx_start, idx_end:  0 15
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  15 30
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  30 45
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  45 60
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  60 75
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  75 90
diff.shape:  (14, 41)
diff.shape:  (14, 40)
Generating counterfactual analysis for patient 40
Generating Counterfactuals...


100%|██████████| 1/1 [00:14<00:00, 14.68s/it]


df_dcf.shape:  (56, 41)
diffs.shape:  (56, 41)
batch_ranges:  [(0, 15), (15, 30), (30, 45), (45, 60)]
idx_start, idx_end:  0 15
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  15 30
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  30 45
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  45 60
diff.shape:  (11, 41)
diff.shape:  (11, 40)
Generating counterfactual analysis for patient 127
Generating Counterfactuals...


100%|██████████| 1/1 [08:57<00:00, 537.37s/it]


df_dcf.shape:  (67, 41)
diffs.shape:  (67, 41)
batch_ranges:  [(0, 15), (15, 30), (30, 45), (45, 60), (60, 75)]
idx_start, idx_end:  0 15
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  15 30
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  30 45
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  45 60
diff.shape:  (15, 41)
diff.shape:  (15, 40)
idx_start, idx_end:  60 75
diff.shape:  (7, 41)
diff.shape:  (7, 40)
Generating counterfactual analysis for patient 182
Generating Counterfactuals...


  0%|          | 0/1 [00:00<?, ?it/s]